In [2]:
# import required libraries

import os
import sys

from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [4]:
python_path = sys.executable
spark = (
    SparkSession.builder
    .appName("disruption_feature_engineering")
    .config(
        "spark.sql.shuffle.partitions",
        "4"
    )
    .config(
        "spark.sql.session.timeZone",
        "Europe/London"
    )
    .config(
        "spark.pyspark.python",
        python_path
    )
    .config(
        "spark.pyspark.driver.python",
        python_path
    )
    .config(
        "spark.executorEnv.PYSPARK_PYTHON",
        python_path
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/15 21:06:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

# joined dataset path

joined_path = (
    project_root
    / "data"
    / "processed"
    / "joined_transport_data"
)

# define the feature output path

feature_path = (
    project_root
    / "data"
    / "processed"
    / "disruption_features"
)

# define the modelling output path

model_path = (
    project_root
    / "data"
    / "processed"
    / "disruption_modelling_data"
)

print("project root:", project_root)
print("joined path exists:", joined_path.exists())
print("feature path:", feature_path)
print("model path:", model_path)

project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
joined path exists: True
feature path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/disruption_features
model path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/disruption_modelling_data


In [6]:
# load the joined transport dataset

joined_df = spark.read.parquet(
    str(joined_path)
)

joined_rows = joined_df.count()

print(
    "joined rows loaded:",
    f"{joined_rows:,}"
)

print(
    "joined partitions:",
    joined_df.rdd.getNumPartitions()
)

joined rows loaded: 9,546
joined partitions: 4


## validate required columns

In [7]:
required_columns = [
    "snapshot_time",
    "recorded_at_time",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "direction_ref",
    "vehicle_ref",
    "dated_vehicle_journey_ref",
    "aimed_start_seconds",
    "latitude",
    "longitude",
    "observation_age_seconds",
    "schedule_time_difference_seconds",
    "match_method",
    "match_quality"
]

# find missing columns

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in joined_df.columns
]

print(
    "missing columns:",
    missing_columns
)

if missing_columns:
    raise ValueError(
        f"missing required columns {missing_columns}"
    )

print(
    "all required columns are available"
)

missing columns: []
all required columns are available


In [8]:
# display match quality counts
match_summary = (
    joined_df
    .groupBy(
        "match_method",
        "match_quality"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

match_summary.show(
    truncate=False
)

+---------------------------+-------------+-----+
|match_method               |match_quality|count|
+---------------------------+-------------+-----+
|exact_operator_line_journey|strong       |6743 |
|fallback_operator_line_time|medium       |2364 |
|exact_operator_line_journey|high         |435  |
|fallback_operator_line_time|provisional  |4    |
+---------------------------+-------------+-----+



## prepare the base dataset

this section creates one consistent snapshot time and recreates the date hour and weekday from that snapshot time

this prevents the same xml snapshot from being divided into different time groups

In [9]:
# create one consistent snapshot time

base_df = (
    joined_df
    .withColumn(
        "event_snapshot_time",
        F.coalesce(
            F.col("snapshot_time"),
            F.col("recorded_at_time")
        )
    )
    .withColumn(
        "observation_date",
        F.to_date(
            "event_snapshot_time"
        )
    )
    .withColumn(
        "observation_hour",
        F.hour(
            "event_snapshot_time"
        )
    )
    .withColumn(
        "observation_day_of_week",
        F.dayofweek(
            "event_snapshot_time"
        )
    )
    .withColumn(
        "exact_match_flag",
        F.when(
            F.col("match_method")
            ==
            "exact_operator_line_journey",
            1
        ).otherwise(0)
    )
    .withColumn(
        "fallback_match_flag",
        F.when(
            F.col("match_method")
            ==
            "fallback_operator_line_time",
            1
        ).otherwise(0)
    )
)

print(
    "base rows:",
    f"{base_df.count():,}"
)

print(
    "missing snapshot times:",
    base_df.filter(
        F.col(
            "event_snapshot_time"
        ).isNull()
    ).count()
)

base rows: 9,546
missing snapshot times: 0


In [10]:
# remove provisional matches
analysis_df = (
    base_df
    .filter(
        F.col("match_quality")
        !=
        "provisional"
    )
)

provisional_rows_removed = (
    base_df.count()
    -
    analysis_df.count()
)

# remove records without important route values

analysis_df = (
    analysis_df
    .filter(
        F.col(
            "event_snapshot_time"
        ).isNotNull()
    )
    .filter(
        F.col(
            "operator_ref"
        ).isNotNull()
    )
    .filter(
        F.col(
            "line_ref"
        ).isNotNull()
    )
    .filter(
        F.col(
            "direction_ref"
        ).isNotNull()
    )
    .filter(
        F.col(
            "vehicle_ref"
        ).isNotNull()
    )
)

analysis_rows = analysis_df.count()

print(
    "rows used for analysis:",
    f"{analysis_rows:,}"
)

print(
    "provisional rows removed:",
    provisional_rows_removed
)

rows used for analysis: 9,542
provisional rows removed: 4


## remove repeated vehicles and journeys

this section keeps one record for each vehicle and journey inside the same route snapshot

In [11]:
# define route snapshot grouping columns

snapshot_keys = [
    "event_snapshot_time",
    "observation_date",
    "observation_hour",
    "observation_day_of_week",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "direction_ref"
]

In [12]:
# remove repeated vehicles inside each route snapshot

vehicle_snapshot_df = (
    analysis_df
    .dropDuplicates(
        [
            "event_snapshot_time",
            "operator_ref",
            "line_ref",
            "direction_ref",
            "vehicle_ref"
        ]
    )
)

print(
    "unique vehicle snapshot rows:",
    f"{vehicle_snapshot_df.count():,}"
)

unique vehicle snapshot rows: 9,542


In [13]:
# create one usable journey key

vehicle_snapshot_df = (
    vehicle_snapshot_df
    .withColumn(
        "analysis_journey_key",
        F.coalesce(
            F.col(
                "dated_vehicle_journey_ref"
            ),
            F.col(
                "vehicle_ref"
            )
        )
    )
)

# keep one record for each journey in a snapshot

journey_snapshot_df = (
    vehicle_snapshot_df
    .dropDuplicates(
        snapshot_keys
        +
        [
            "analysis_journey_key"
        ]
    )
)

print(
    "unique journey snapshot rows:",
    f"{journey_snapshot_df.count():,}"
)

[Stage 28:>                                                         (0 + 4) / 4]

unique journey snapshot rows: 9,510


In [14]:
# order journeys inside each route snapshot

departure_window = (
    Window
    .partitionBy(
        *snapshot_keys
    )
    .orderBy(
        F.col(
            "aimed_start_seconds"
        ).asc_nulls_last()
    )
)

# find the previous aimed departure

spacing_df = (
    journey_snapshot_df
    .withColumn(
        "previous_aimed_start_seconds",
        F.lag(
            "aimed_start_seconds"
        ).over(
            departure_window
        )
    )
)

# calculate departure spacing

spacing_df = (
    spacing_df
    .withColumn(
        "raw_departure_spacing_seconds",
        F.col(
            "aimed_start_seconds"
        )
        -
        F.col(
            "previous_aimed_start_seconds"
        )
    )
    .withColumn(
        "departure_spacing_seconds",
        F.when(
            (
                F.col(
                    "raw_departure_spacing_seconds"
                )
                >=
                0
            )
            &
            (
                F.col(
                    "raw_departure_spacing_seconds"
                )
                <=
                14400
            ),
            F.col(
                "raw_departure_spacing_seconds"
            )
        )
    )
)

print(
    "departure spacing created"
)

departure spacing created


In [15]:
# create route level snapshot features

route_snapshot_df = (
    spacing_df
    .groupBy(
        *snapshot_keys
    )
    .agg(
        F.count("*").alias(
            "observation_count"
        ),
        F.countDistinct(
            "vehicle_ref"
        ).alias(
            "observed_vehicle_count"
        ),
        F.countDistinct(
            "analysis_journey_key"
        ).alias(
            "observed_journey_count"
        ),
        F.avg(
            "observation_age_seconds"
        ).alias(
            "average_observation_age_seconds"
        ),
        F.max(
            "observation_age_seconds"
        ).alias(
            "maximum_observation_age_seconds"
        ),
        F.avg(
            "schedule_time_difference_seconds"
        ).alias(
            "average_match_time_difference_seconds"
        ),
        F.max(
            "schedule_time_difference_seconds"
        ).alias(
            "maximum_match_time_difference_seconds"
        ),
        F.avg(
            "exact_match_flag"
        ).alias(
            "exact_match_rate"
        ),
        F.avg(
            "fallback_match_flag"
        ).alias(
            "fallback_match_rate"
        ),
        F.count(
            "departure_spacing_seconds"
        ).alias(
            "valid_spacing_count"
        ),
        F.avg(
            "departure_spacing_seconds"
        ).alias(
            "average_departure_spacing_seconds"
        ),
        F.max(
            "departure_spacing_seconds"
        ).alias(
            "maximum_departure_spacing_seconds"
        ),
        F.stddev_samp(
            "departure_spacing_seconds"
        ).alias(
            "departure_spacing_std_seconds"
        ),
        F.min(
            "latitude"
        ).alias(
            "minimum_latitude"
        ),
        F.max(
            "latitude"
        ).alias(
            "maximum_latitude"
        ),
        F.min(
            "longitude"
        ).alias(
            "minimum_longitude"
        ),
        F.max(
            "longitude"
        ).alias(
            "maximum_longitude"
        )
    )
)

route_snapshot_rows = (
    route_snapshot_df.count()
)

print(
    "route snapshot rows:",
    f"{route_snapshot_rows:,}"
)

[Stage 38:>                                                         (0 + 4) / 4]

route snapshot rows: 1,926


In [16]:
# display selected route snapshot features

route_snapshot_df.select(
    "event_snapshot_time",
    "operator_ref",
    "line_ref",
    "direction_ref",
    "observed_vehicle_count",
    "observed_journey_count",
    F.round(
        "exact_match_rate",
        2
    ).alias(
        "exact_match_rate"
    ),
    F.round(
        "fallback_match_rate",
        2
    ).alias(
        "fallback_match_rate"
    ),
    "maximum_departure_spacing_seconds"
).show(
    10,
    truncate=False
)

[Stage 57:>                                                         (0 + 1) / 1]

+-----------------------+------------+--------+-------------+----------------------+----------------------+----------------+-------------------+---------------------------------+
|event_snapshot_time    |operator_ref|line_ref|direction_ref|observed_vehicle_count|observed_journey_count|exact_match_rate|fallback_match_rate|maximum_departure_spacing_seconds|
+-----------------------+------------+--------+-------------+----------------------+----------------------+----------------+-------------------+---------------------------------+
|2026-07-11 18:48:58.184|TCVW        |12X     |inbound      |5                     |5                     |1.0             |0.0                |9000                             |
|2026-07-11 18:48:58.184|TCVW        |17A     |outbound     |4                     |4                     |0.25            |0.75               |12840                            |
|2026-07-11 18:48:58.184|TCVW        |20      |inbound      |6                     |6                    

In [17]:
# create time and data quality features

route_snapshot_df = (
    route_snapshot_df
    .withColumn(
        "is_peak_hour",
        F.when(
            F.col(
                "observation_hour"
            ).isin(
                7,
                8,
                9,
                16,
                17,
                18
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_weekend",
        F.when(
            F.col(
                "observation_day_of_week"
            ).isin(
                1,
                7
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "hour_sin",
        F.sin(
            F.col(
                "observation_hour"
            )
            *
            F.lit(
                2
                *
                3.141592653589793
                /
                24
            )
        )
    )
    .withColumn(
        "hour_cos",
        F.cos(
            F.col(
                "observation_hour"
            )
            *
            F.lit(
                2
                *
                3.141592653589793
                /
                24
            )
        )
    )
    .withColumn(
        "data_quality_flag",
        F.when(
            F.col(
                "exact_match_rate"
            )
            <
            0.50,
            1
        ).otherwise(0)
    )
)

print(
    "time and data quality features created"
)

time and data quality features created


In [18]:
# order snapshots for each route and direction

route_order_window = (
    Window
    .partitionBy(
        "operator_ref",
        "line_ref",
        "direction_ref"
    )
    .orderBy(
        F.col(
            "event_snapshot_time"
        )
    )
)

# create a window containing earlier snapshots only

history_window = (
    route_order_window
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)

# calculate historical route values

feature_df = (
    route_snapshot_df
    .withColumn(
        "historical_vehicle_count",
        F.avg(
            "observed_vehicle_count"
        ).over(
            history_window
        )
    )
    .withColumn(
        "historical_spacing_seconds",
        F.avg(
            "maximum_departure_spacing_seconds"
        ).over(
            history_window
        )
    )
    .withColumn(
        "historical_snapshot_count",
        F.count(
            F.lit(1)
        ).over(
            history_window
        )
    )
    .withColumn(
        "previous_vehicle_count",
        F.lag(
            "observed_vehicle_count"
        ).over(
            route_order_window
        )
    )
    .withColumn(
        "previous_snapshot_time",
        F.lag(
            "event_snapshot_time"
        ).over(
            route_order_window
        )
    )
)

print(
    "historical route features created"
)

historical route features created


In [19]:
# prepare baseline values

feature_df = (
    feature_df
    .withColumn(
        "vehicle_count_baseline",
        F.coalesce(
            F.col(
                "historical_vehicle_count"
            ),
            F.col(
                "observed_vehicle_count"
            ).cast(
                "double"
            )
        )
    )
    .withColumn(
        "spacing_baseline_seconds",
        F.coalesce(
            F.col(
                "historical_spacing_seconds"
            ),
            F.col(
                "maximum_departure_spacing_seconds"
            ),
            F.lit(
                0.0
            )
        )
    )
)

print(
    "baseline values prepared"
)

baseline values prepared


In [20]:
# calculate route activity and spacing ratios

feature_df = (
    feature_df
    .withColumn(
        "activity_ratio",
        F.when(
            F.col(
                "vehicle_count_baseline"
            )
            >
            0,
            F.col(
                "observed_vehicle_count"
            )
            /
            F.col(
                "vehicle_count_baseline"
            )
        ).otherwise(
            F.lit(
                1.0
            )
        )
    )
    .withColumn(
        "spacing_ratio",
        F.when(
            (
                F.col(
                    "spacing_baseline_seconds"
                )
                >
                0
            )
            &
            (
                F.col(
                    "maximum_departure_spacing_seconds"
                ).isNotNull()
            ),
            F.col(
                "maximum_departure_spacing_seconds"
            )
            /
            F.col(
                "spacing_baseline_seconds"
            )
        ).otherwise(
            F.lit(
                1.0
            )
        )
    )
)

print(
    "route ratios created"
)

route ratios created


In [21]:
# calculate the time from the previous snapshot

feature_df = (
    feature_df
    .withColumn(
        "minutes_since_previous_snapshot",
        (
            F.unix_timestamp(
                "event_snapshot_time"
            )
            -
            F.unix_timestamp(
                "previous_snapshot_time"
            )
        )
        /
        F.lit(
            60.0
        )
    )
)

# calculate operational change features

feature_df = (
    feature_df
    .withColumn(
        "activity_shortfall",
        F.greatest(
            F.lit(
                0.0
            ),
            F.lit(
                1.0
            )
            -
            F.col(
                "activity_ratio"
            )
        )
    )
    .withColumn(
        "spacing_excess",
        F.greatest(
            F.lit(
                0.0
            ),
            F.col(
                "spacing_ratio"
            )
            -
            F.lit(
                1.0
            )
        )
    )
    .withColumn(
        "previous_vehicle_drop",
        F.when(
            (
                F.col(
                    "previous_vehicle_count"
                )
                >
                0
            )
            &
            (
                F.col(
                    "minutes_since_previous_snapshot"
                )
                >=
                1
            )
            &
            (
                F.col(
                    "minutes_since_previous_snapshot"
                )
                <=
                180
            ),
            F.greatest(
                F.lit(
                    0.0
                ),
                (
                    F.col(
                        "previous_vehicle_count"
                    )
                    -
                    F.col(
                        "observed_vehicle_count"
                    )
                )
                /
                F.col(
                    "previous_vehicle_count"
                )
            )
        ).otherwise(
            F.lit(
                0.0
            )
        )
    )
    .withColumn(
        "single_vehicle_flag",
        F.when(
            (
                F.col(
                    "observed_vehicle_count"
                )
                ==
                1
            )
            &
            (
                F.col(
                    "vehicle_count_baseline"
                )
                >=
                2
            ),
            1
        ).otherwise(0)
    )
)

print(
    "operational change features created"
)

operational change features created


In [22]:
# create the operational irregularity score

feature_df = (
    feature_df
    .withColumn(
        "operational_irregularity_score",
        (
            F.lit(
                0.50
            )
            *
            F.least(
                F.col(
                    "activity_shortfall"
                ),
                F.lit(
                    1.0
                )
            )
        )
        +
        (
            F.lit(
                0.30
            )
            *
            F.least(
                F.col(
                    "spacing_excess"
                )
                /
                F.lit(
                    2.0
                ),
                F.lit(
                    1.0
                )
            )
        )
        +
        (
            F.lit(
                0.20
            )
            *
            F.least(
                F.col(
                    "previous_vehicle_drop"
                ),
                F.lit(
                    1.0
                )
            )
        )
    )
)

print(
    "operational irregularity score created"
)


operational irregularity score created


In [25]:
# inspect the irregularity score distribution

feature_df.select(
    F.round(
        F.min(
            "operational_irregularity_score"
        ),
        4
    ).alias(
        "minimum_score"
    ),
    F.round(
        F.avg(
            "operational_irregularity_score"
        ),
        4
    ).alias(
        "average_score"
    ),
    F.round(
        F.max(
            "operational_irregularity_score"
        ),
        4
    ).alias(
        "maximum_score"
    )
).show(
    truncate=False
)

[Stage 118:==========================================>              (3 + 1) / 4]

+-------------+-------------+-------------+
|minimum_score|average_score|maximum_score|
+-------------+-------------+-------------+
|0.0          |0.0869       |0.4905       |
+-------------+-------------+-------------+



In [26]:
# calculate thresholds from the score distribution

severity_thresholds = (
    feature_df
    .approxQuantile(
        "operational_irregularity_score",
        [
            0.70,
            0.90
        ],
        0.01
    )
)

medium_threshold = severity_thresholds[0]
high_threshold = severity_thresholds[1]

print(
    "medium threshold:",
    medium_threshold
)

print(
    "high threshold:",
    high_threshold
)


[Stage 139:==========================================>              (3 + 1) / 4]

medium threshold: 0.125
high threshold: 0.2649560117302052


## severity labels

the operational irregularity score is converted into low medium and high severity categories

In [27]:
# create low medium and high severity labels

feature_df = (
    feature_df
    .withColumn(
        "severity_label",
        F.when(
            F.col(
                "operational_irregularity_score"
            )
            >=
            high_threshold,
            2
        )
        .when(
            F.col(
                "operational_irregularity_score"
            )
            >=
            medium_threshold,
            1
        )
        .otherwise(
            0
        )
    )
    .withColumn(
        "severity_name",
        F.when(
            F.col(
                "severity_label"
            )
            ==
            2,
            "high"
        )
        .when(
            F.col(
                "severity_label"
            )
            ==
            1,
            "medium"
        )
        .otherwise(
            "low"
        )
    )
)

print(
    "severity labels created"
)

severity labels created


In [28]:
# display severity class distribution

severity_summary = (
    feature_df
    .groupBy(
        "severity_label",
        "severity_name"
    )
    .count()
    .orderBy(
        "severity_label"
    )
)

severity_summary.show(
    truncate=False
)

print(
    "number of severity classes:",
    severity_summary.count()
)

+--------------+-------------+-----+
|severity_label|severity_name|count|
+--------------+-------------+-----+
|0             |low          |1324 |
|1             |medium       |391  |
|2             |high         |211  |
+--------------+-------------+-----+

number of severity classes: 3


In [29]:
model_df = (
    feature_df
    .withColumn(
        "next_snapshot_time",
        F.lead(
            "event_snapshot_time"
        ).over(
            route_order_window
        )
    )
    .withColumn(
        "target_severity_label",
        F.lead(
            "severity_label"
        ).over(
            route_order_window
        )
    )
    .withColumn(
        "target_severity_name",
        F.lead(
            "severity_name"
        ).over(
            route_order_window
        )
    )
)

# calculate the time to the next snapshot

model_df = (
    model_df
    .withColumn(
        "minutes_to_next_snapshot",
        (
            F.unix_timestamp(
                "next_snapshot_time"
            )
            -
            F.unix_timestamp(
                "event_snapshot_time"
            )
        )
        /
        F.lit(
            60.0
        )
    )
)

# remove rows without a future target

model_df = (
    model_df
    .filter(
        F.col(
            "target_severity_label"
        ).isNotNull()
    )
)

model_rows_before_filter = model_df.count()

print(
    "rows with a prediction target:",
    f"{model_rows_before_filter:,}"
)

[Stage 203:============================>                            (2 + 2) / 4]

rows with a prediction target: 1,646


In [30]:
#  prediction horizon groups
model_df = (
    model_df
    .withColumn(
        "prediction_horizon",
        F.when(
            F.col(
                "minutes_to_next_snapshot"
            )
            <
            1,
            "under 1 minute"
        )
        .when(
            F.col(
                "minutes_to_next_snapshot"
            )
            <=
            30,
            "1 to 30 minutes"
        )
        .when(
            F.col(
                "minutes_to_next_snapshot"
            )
            <=
            60,
            "31 to 60 minutes"
        )
        .when(
            F.col(
                "minutes_to_next_snapshot"
            )
            <=
            120,
            "61 to 120 minutes"
        )
        .when(
            F.col(
                "minutes_to_next_snapshot"
            )
            <=
            180,
            "121 to 180 minutes"
        )
        .otherwise(
            "more than 180 minutes"
        )
    )
)

print(
    "prediction horizon created"
)

prediction horizon created


In [31]:
# mark rows suitable for model training

model_df = (
    model_df
    .withColumn(
        "model_eligible_flag",
        F.when(
            (
                F.col(
                    "minutes_to_next_snapshot"
                )
                >=
                1
            )
            &
            (
                F.col(
                    "minutes_to_next_snapshot"
                )
                <=
                180
            )
            &
            (
                F.col(
                    "historical_snapshot_count"
                )
                >=
                1
            ),
            1
        ).otherwise(
            0
        )
    )
)

model_eligible_rows = (
    model_df
    .filter(
        F.col(
            "model_eligible_flag"
        )
        ==
        1
    )
    .count()
)

print(
    "model eligible rows:",
    f"{model_eligible_rows:,}"
)

model eligible rows: 239


In [32]:
# display prediction horizon distribution

horizon_summary = (
    model_df
    .groupBy(
        "prediction_horizon"
    )
    .count()
    .orderBy(
        F.desc(
            "count"
        )
    )
)

horizon_summary.show(
    truncate=False
)

+---------------------+-----+
|prediction_horizon   |count|
+---------------------+-----+
|more than 180 minutes|1152 |
|under 1 minute       |250  |
|61 to 120 minutes    |244  |
+---------------------+-----+



In [33]:
# display prediction time statistics

model_df.select(
    F.round(
        F.min(
            "minutes_to_next_snapshot"
        ),
        2
    ).alias(
        "minimum_minutes"
    ),
    F.round(
        F.avg(
            "minutes_to_next_snapshot"
        ),
        2
    ).alias(
        "average_minutes"
    ),
    F.round(
        F.max(
            "minutes_to_next_snapshot"
        ),
        2
    ).alias(
        "maximum_minutes"
    )
).show(
    truncate=False
)

+---------------+---------------+---------------+
|minimum_minutes|average_minutes|maximum_minutes|
+---------------+---------------+---------------+
|0.1            |619.23         |2898.22        |
+---------------+---------------+---------------+



## target class distribution

this section checks the low medium and high severity classes that the machine learning models will predict

In [34]:
# display target severity distribution

target_summary = (
    model_df
    .groupBy(
        "target_severity_label",
        "target_severity_name"
    )
    .count()
    .orderBy(
        "target_severity_label"
    )
)

target_summary.show(
    truncate=False
)

target_class_count = (
    target_summary.count()
)

print(
    "number of target classes:",
    target_class_count
)

if target_class_count < 3:
    print(
        "warning target data does not contain all three classes"
    )

+---------------------+--------------------+-----+
|target_severity_label|target_severity_name|count|
+---------------------+--------------------+-----+
|0                    |low                 |1044 |
|1                    |medium              |391  |
|2                    |high                |211  |
+---------------------+--------------------+-----+

number of target classes: 3


In [35]:
# display target classes for eligible model rows

eligible_target_summary = (
    model_df
    .filter(
        F.col(
            "model_eligible_flag"
        )
        ==
        1
    )
    .groupBy(
        "target_severity_label",
        "target_severity_name"
    )
    .count()
    .orderBy(
        "target_severity_label"
    )
)

eligible_target_summary.show(
    truncate=False
)

eligible_class_count = (
    eligible_target_summary.count()
)

print(
    "eligible target classes:",
    eligible_class_count
)

+---------------------+--------------------+-----+
|target_severity_label|target_severity_name|count|
+---------------------+--------------------+-----+
|0                    |low                 |173  |
|1                    |medium              |46   |
|2                    |high                |20   |
+---------------------+--------------------+-----+

eligible target classes: 3


## final modelling dataset

this section keeps the columns required for exploratory analysis model training and evaluation

In [36]:
# select final modelling columns

model_columns = [
    "event_snapshot_time",
    "next_snapshot_time",
    "minutes_to_next_snapshot",
    "prediction_horizon",
    "model_eligible_flag",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "direction_ref",
    "observation_date",
    "observation_hour",
    "observation_day_of_week",
    "is_peak_hour",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "observed_vehicle_count",
    "observed_journey_count",
    "average_observation_age_seconds",
    "maximum_observation_age_seconds",
    "average_match_time_difference_seconds",
    "maximum_match_time_difference_seconds",
    "exact_match_rate",
    "fallback_match_rate",
    "data_quality_flag",
    "valid_spacing_count",
    "average_departure_spacing_seconds",
    "maximum_departure_spacing_seconds",
    "departure_spacing_std_seconds",
    "historical_vehicle_count",
    "historical_spacing_seconds",
    "historical_snapshot_count",
    "previous_vehicle_count",
    "minutes_since_previous_snapshot",
    "vehicle_count_baseline",
    "spacing_baseline_seconds",
    "activity_ratio",
    "spacing_ratio",
    "activity_shortfall",
    "spacing_excess",
    "previous_vehicle_drop",
    "single_vehicle_flag",
    "operational_irregularity_score",
    "severity_label",
    "severity_name",
    "target_severity_label",
    "target_severity_name"
]

model_df = (
    model_df
    .select(
        *model_columns
    )
)

print(
    "final modelling columns:",
    len(model_df.columns)
)

final modelling columns: 47


In [37]:
# display selected modelling features

model_df.select(
    "event_snapshot_time",
    "next_snapshot_time",
    F.round(
        "minutes_to_next_snapshot",
        2
    ).alias(
        "minutes_to_next_snapshot"
    ),
    "line_ref",
    "direction_ref",
    "observed_vehicle_count",
    F.round(
        "activity_ratio",
        2
    ).alias(
        "activity_ratio"
    ),
    F.round(
        "spacing_ratio",
        2
    ).alias(
        "spacing_ratio"
    ),
    F.round(
        "operational_irregularity_score",
        3
    ).alias(
        "irregularity_score"
    ),
    "severity_name",
    "target_severity_name",
    "model_eligible_flag"
).show(
    10,
    truncate=False
)

+-----------------------+-----------------------+------------------------+--------+-------------+----------------------+--------------+-------------+------------------+-------------+--------------------+-------------------+
|event_snapshot_time    |next_snapshot_time     |minutes_to_next_snapshot|line_ref|direction_ref|observed_vehicle_count|activity_ratio|spacing_ratio|irregularity_score|severity_name|target_severity_name|model_eligible_flag|
+-----------------------+-----------------------+------------------------+--------+-------------+----------------------+--------------+-------------+------------------+-------------+--------------------+-------------------+
|2026-07-11 18:48:58.184|2026-07-12 05:16:59.303|628.02                  |11      |inbound      |6                     |1.0           |1.0          |0.0               |low          |low                 |0                  |
|2026-07-12 05:16:59.303|2026-07-13 11:39:54.231|1822.92                 |11      |inbound      |9      

In [38]:
# define important modelling columns

important_columns = [
    "event_snapshot_time",
    "next_snapshot_time",
    "minutes_to_next_snapshot",
    "observation_hour",
    "is_peak_hour",
    "is_weekend",
    "observed_vehicle_count",
    "observed_journey_count",
    "exact_match_rate",
    "fallback_match_rate",
    "activity_ratio",
    "spacing_ratio",
    "activity_shortfall",
    "spacing_excess",
    "previous_vehicle_drop",
    "operational_irregularity_score",
    "severity_label",
    "target_severity_label",
    "model_eligible_flag"
]

# count missing values in important columns

null_summary = (
    model_df
    .select(
        [
            F.sum(
                F.when(
                    F.col(
                        column_name
                    ).isNull(),
                    1
                ).otherwise(
                    0
                )
            ).alias(
                column_name
            )
            for column_name
            in important_columns
        ]
    )
)

null_summary.show(
    truncate=False
)

+-------------------+------------------+------------------------+----------------+------------+----------+----------------------+----------------------+----------------+-------------------+--------------+-------------+------------------+--------------+---------------------+------------------------------+--------------+---------------------+-------------------+
|event_snapshot_time|next_snapshot_time|minutes_to_next_snapshot|observation_hour|is_peak_hour|is_weekend|observed_vehicle_count|observed_journey_count|exact_match_rate|fallback_match_rate|activity_ratio|spacing_ratio|activity_shortfall|spacing_excess|previous_vehicle_drop|operational_irregularity_score|severity_label|target_severity_label|model_eligible_flag|
+-------------------+------------------+------------------------+----------------+------------+----------+----------------------+----------------------+----------------+-------------------+--------------+-------------+------------------+--------------+---------------------+

In [39]:
# count invalid prediction times

invalid_prediction_rows = (
    model_df
    .filter(
        (
            F.col(
                "minutes_to_next_snapshot"
            ).isNull()
        )
        |
        (
            F.col(
                "minutes_to_next_snapshot"
            )
            <=
            0
        )
    )
    .count()
)

print(
    "invalid prediction time rows:",
    invalid_prediction_rows
)

invalid prediction time rows: 0


In [40]:
# repartition and cache feature data

feature_df = (
    feature_df
    .repartition(
        4,
        "line_ref"
    )
)

feature_df.cache()

feature_rows = feature_df.count()

print(
    "feature rows:",
    f"{feature_rows:,}"
)

print(
    "feature partitions:",
    feature_df.rdd.getNumPartitions()
)

26/07/15 22:08:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

feature rows: 1,926
feature partitions: 4


In [41]:
# repartition and cache modelling data

model_df = (
    model_df
    .repartition(
        4,
        "line_ref"
    )
)

model_df.cache()

model_rows = model_df.count()

model_eligible_rows = (
    model_df
    .filter(
        F.col(
            "model_eligible_flag"
        )
        ==
        1
    )
    .count()
)

print(
    "modelling rows:",
    f"{model_rows:,}"
)

print(
    "model eligible rows:",
    f"{model_eligible_rows:,}"
)

print(
    "modelling partitions:",
    model_df.rdd.getNumPartitions()
)

modelling rows: 1,646
model eligible rows: 239
modelling partitions: 4


## save processed datasets

this section saves the route snapshot features and modelling dataset as parquet files

In [42]:
# save feature dataset

(
    feature_df
    .write
    .mode(
        "overwrite"
    )
    .parquet(
        str(
            feature_path
        )
    )
)

print(
    "feature dataset saved"
)

[Stage 572:>                                                        (0 + 4) / 4]

feature dataset saved


In [43]:
# save modelling dataset

(
    model_df
    .write
    .mode(
        "overwrite"
    )
    .parquet(
        str(
            model_path
        )
    )
)

print(
    "modelling dataset saved"
)

[Stage 580:>                                                        (0 + 4) / 4]

modelling dataset saved


In [44]:
# reload saved feature dataset

saved_feature_df = (
    spark.read.parquet(
        str(
            feature_path
        )
    )
)

# reload saved modelling dataset

saved_model_df = (
    spark.read.parquet(
        str(
            model_path
        )
    )
)

print(
    "saved datasets reloaded"
)

saved datasets reloaded


In [45]:
# verify saved feature dataset

saved_feature_rows = (
    saved_feature_df.count()
)

saved_feature_partitions = (
    saved_feature_df
    .rdd
    .getNumPartitions()
)

print(
    "saved feature rows:",
    f"{saved_feature_rows:,}"
)

print(
    "saved feature partitions:",
    saved_feature_partitions
)

if saved_feature_rows != feature_rows:
    raise ValueError(
        "saved feature row count does not match"
    )

saved feature rows: 1,926
saved feature partitions: 4


In [46]:
# verify saved modelling dataset

saved_model_rows = (
    saved_model_df.count()
)

saved_model_partitions = (
    saved_model_df
    .rdd
    .getNumPartitions()
)

saved_eligible_rows = (
    saved_model_df
    .filter(
        F.col(
            "model_eligible_flag"
        )
        ==
        1
    )
    .count()
)

print(
    "saved modelling rows:",
    f"{saved_model_rows:,}"
)

print(
    "saved eligible rows:",
    f"{saved_eligible_rows:,}"
)

print(
    "saved modelling partitions:",
    saved_model_partitions
)

if saved_model_rows != model_rows:
    raise ValueError(
        "saved modelling row count does not match"
    )

saved modelling rows: 1,646
saved eligible rows: 239
saved modelling partitions: 4


In [47]:
# verify saved target classes

saved_model_df.groupBy(
    "target_severity_label",
    "target_severity_name"
).count().orderBy(
    "target_severity_label"
).show(
    truncate=False
)

saved_target_class_count = (
    saved_model_df
    .select(
        "target_severity_label"
    )
    .distinct()
    .count()
)

print(
    "saved target classes:",
    saved_target_class_count
)

if saved_target_class_count < 3:
    print(
        "warning saved model data does not contain all three classes"
    )

+---------------------+--------------------+-----+
|target_severity_label|target_severity_name|count|
+---------------------+--------------------+-----+
|0                    |low                 |1044 |
|1                    |medium              |391  |
|2                    |high                |211  |
+---------------------+--------------------+-----+

saved target classes: 3


In [48]:
# complete final verification

if saved_feature_rows != feature_rows:
    raise ValueError(
        "feature verification failed"
    )

if saved_model_rows != model_rows:
    raise ValueError(
        "modelling verification failed"
    )

if saved_feature_partitions < 4:
    raise ValueError(
        "feature dataset has fewer than four partitions"
    )

if saved_model_partitions < 4:
    raise ValueError(
        "modelling dataset has fewer than four partitions"
    )

if saved_target_class_count < 3:
    raise ValueError(
        "modelling target has fewer than three classes"
    )

print(
    "feature engineering verification successful"
)

feature engineering verification successful
